# 🚀 ProvenanceLab: Train HiDDeN-32 on Google Colab (Free GPU)

This notebook trains your **HiDDeN-32 Neural Watermarking Model** using a free GPU.

### Key Invariants Preserved:
1. **Protects Your Blockchain & ZK Keys:** The 64-to-32 Linear Head (`fc`) is **strictly frozen** so your Zero-Knowledge Halo2 circuit and Ethereum Sepolia smart contract remain 100% compatible.
2. **Real Generalization:** Trains on clean real-world photos with random 32-bit payloads on every batch.
3. **Fast GPU Training:** Takes ~3–5 minutes on a free Google Colab T4 GPU.

### Step 1: Check GPU Acceleration
Make sure your Colab runtime is set to **GPU** (`Runtime` -> `Change runtime type` -> select `T4 GPU`).

In [ ]:
import torch
print('PyTorch Version:', torch.__version__)
if torch.cuda.is_available():
    print('✅ SUCCESS: Using GPU ->', torch.cuda.get_device_name(0))
    device = torch.device('cuda')
else:
    print('⚠️ WARNING: Running on CPU. Go to Runtime -> Change runtime type -> T4 GPU')
    device = torch.device('cpu')

### Step 2: Define the HiDDeN-32 Neural Architecture

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class ConvBNRelu(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.relu(self.conv(x))

class HiddenEncoder(nn.Module):
    def __init__(self, message_bits: int = 32, channels: int = 64, residual_scale: float = 0.10):
        super().__init__()
        self.message_bits = message_bits
        self.residual_scale = residual_scale
        self.conv1 = ConvBNRelu(3, channels)
        self.conv2 = ConvBNRelu(channels, channels)
        self.conv3 = ConvBNRelu(channels + message_bits, channels)
        self.conv4 = ConvBNRelu(channels, channels)
        self.residual_conv = nn.Conv2d(channels, 3, kernel_size=1)

    def forward(self, image: torch.Tensor, message: torch.Tensor) -> torch.Tensor:
        B, _, H, W = image.shape
        expanded_msg = message.unsqueeze(-1).unsqueeze(-1).expand(B, self.message_bits, H, W)
        x = self.conv1(image)
        x = self.conv2(x)
        joint = torch.cat([x, expanded_msg], dim=1)
        x = self.conv3(joint)
        x = self.conv4(x)
        delta = torch.tanh(self.residual_conv(x)) * self.residual_scale
        return torch.clamp(image + delta, -1.0, 1.0)

class HiddenDecoder(nn.Module):
    def __init__(self, message_bits: int = 32, channels: int = 64):
        super().__init__()
        self.message_bits = message_bits
        self.conv1 = ConvBNRelu(3, channels)
        self.conv2 = ConvBNRelu(channels, channels)
        self.conv3 = ConvBNRelu(channels, channels)
        self.conv4 = ConvBNRelu(channels, channels)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(channels, message_bits)

    def extract_features(self, image: torch.Tensor) -> torch.Tensor:
        x = self.conv1(image)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        return self.pool(x).flatten(1)

    def forward(self, image: torch.Tensor) -> torch.Tensor:
        pooled = self.extract_features(image)
        return self.fc(pooled)

class HiddenModel(nn.Module):
    def __init__(self, message_bits: int = 32, channels: int = 64, residual_scale: float = 0.10):
        super().__init__()
        self.encoder = HiddenEncoder(message_bits, channels, residual_scale)
        self.decoder = HiddenDecoder(message_bits, channels)

print('✅ Neural Architecture defined successfully.')

### Step 3: Download Clean Photos (COCO Dataset)

In [ ]:
import os, urllib.request, zipfile
from pathlib import Path

# Download Ultralytics COCO-128 clean photos dataset
dataset_url = 'https://github.com/ultralytics/yolov5/releases/download/v1.0/coco128.zip'
os.makedirs('dataset', exist_ok=True)

print('Downloading clean dataset...')
urllib.request.urlretrieve(dataset_url, 'coco128.zip')
with zipfile.ZipFile('coco128.zip', 'r') as zip_ref:
    zip_ref.extractall('dataset')

# Gather all image paths
img_paths = sorted([str(p) for p in Path('dataset').rglob('*') if p.suffix.lower() in ['.jpg', '.png', '.jpeg']])
print(f'✅ Extracted {len(img_paths)} clean images!')

### Step 4: The Golden Rule (Freeze the ZK Linear Head)
We initialize the model and **lock** `fc = nn.Linear(64, 32)`. This protects your Zero-Knowledge Halo2 circuit and smart contracts.

In [ ]:
import numpy as np
from PIL import Image

model = HiddenModel(message_bits=32, channels=64, residual_scale=0.10).to(device)

# -------------------------------------------------------------------------
# THE GOLDEN RULE: FREEZE LINEAR HEAD
# -------------------------------------------------------------------------
model.decoder.fc.weight.requires_grad = False
model.decoder.fc.bias.requires_grad = False
print('🔒 Linear Head (Linear(64, 32)) is strictly LOCKED (Protects ZK & Blockchain)!')

# Load images into memory
RESOLUTION = 128
tensors = []
print(f'Preprocessing images to {RESOLUTION}x{RESOLUTION}...')
for p in img_paths:
    im = Image.open(p).convert('RGB').resize((RESOLUTION, RESOLUTION), Image.Resampling.BICUBIC)
    arr = np.array(im, dtype=np.float32) / 127.5 - 1.0
    tensors.append(torch.from_numpy(arr).permute(2, 0, 1))

train_imgs = tensors[:int(len(tensors) * 0.9)]
val_imgs = tensors[int(len(tensors) * 0.9):]
print(f'✅ Ready: {len(train_imgs)} Training Images | {len(val_imgs)} Validation Images')

### Step 5: Train the Model on GPU!
Watch the **Bit Error Rate (BER)** drop towards 0% while **Visual Quality (PSNR)** stays above 30 dB.

In [ ]:
import torch.optim as optim
import random, math, time

EPOCHS = 50
BATCH_SIZE = 16
LEARNING_RATE = 0.001

trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.Adam(trainable, lr=LEARNING_RATE)

print(f'Starting GPU training for {EPOCHS} epochs...')
best_ber = 1.0
start_t = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    random.shuffle(train_imgs)
    
    for i in range(0, len(train_imgs), BATCH_SIZE):
        batch = train_imgs[i : i + BATCH_SIZE]
        if len(batch) < 2: continue
        
        imgs = torch.stack(batch).to(device)
        B = imgs.shape[0]
        # Random 32-bit payloads
        msgs = torch.randint(0, 2, (B, 32), dtype=torch.float32, device=device)
        
        optimizer.zero_grad()
        wm_imgs = model.encoder(imgs, msgs)
        pred_logits = model.decoder(wm_imgs)
        
        loss_msg = F.binary_cross_entropy_with_logits(pred_logits, msgs)
        loss_img = F.mse_loss(wm_imgs, imgs)
        total_loss = 15.0 * loss_msg + 2.0 * loss_img
        
        total_loss.backward()
        optimizer.step()
        
    # Validation step
    model.eval()
    with torch.no_grad():
        val_batch = torch.stack(val_imgs).to(device)
        val_msgs = torch.randint(0, 2, (len(val_imgs), 32), dtype=torch.float32, device=device)
        
        val_wm = model.encoder(val_batch, val_msgs)
        val_logits = model.decoder(val_wm)
        
        preds = (val_logits >= 0.0).float()
        val_ber = (preds != val_msgs).float().mean().item()
        val_mse = F.mse_loss(val_wm, val_batch).item()
        val_psnr = 10.0 * math.log10(4.0 / max(1e-10, val_mse))
        
    print(f'Epoch [{epoch:2d}/{EPOCHS:2d}] | BER: {val_ber * 100:5.1f}% | PSNR: {val_psnr:5.1f} dB | Elapsed: {time.time() - start_t:.0f}s')
    
    if val_ber < best_ber:
        best_ber = val_ber
        torch.save({
            'encoder': model.encoder.state_dict(),
            'decoder': model.decoder.state_dict(),
            'metadata': {
                'architecture': 'HiDDeN-32bit-Colab',
                'best_ber': best_ber,
                'psnr': val_psnr,
                'linear_head_frozen': True
            }
        }, 'hidden_32bit_trained.pt')
        print(f'  ⭐ Saved new best checkpoint (BER: {val_ber * 100:.1f}%)')

print('\n🎉 Training Complete! Best BER:', f'{best_ber * 100:.1f}%')

### Step 6: Download the Trained Checkpoint
Run this cell to download `hidden_32bit_trained.pt` to your computer, then move it to your project's `models/checkpoints/` folder!

In [ ]:
from google.colab import files
files.download('hidden_32bit_trained.pt')
print('✅ Download started! Place hidden_32bit_trained.pt into models/checkpoints/ in your project.')